# SSCFLP — main
**Todo o código está neste notebook:** leitura Parquet, heurísticas, solver LP/MIP, auditoria, exportação e gráficos. Não há arquivos `.py` para importar.

No Kaggle, crie um Dataset privado com **somente** `saiwa.parquet`, `holmberg.parquet`, `beasley.parquet` e `origens.json`, todos juntos. Importe `main.ipynb`, anexe o Dataset e execute as células em ordem. CPU basta. Localmente, abra este mesmo notebook na raiz do projeto.

Requer Python 3.11+ e PyArrow. Gráficos usam Matplotlib; o solver usa PySCIPOpt/SCIP. Se necessário, execute `%pip install pyarrow matplotlib`; para o solver, `%pip install pyscipopt==6.3.0`. No Kaggle, instalações exigem Internet. **Heurísticas, solver e gráficos ficam desligados por padrão.**

In [ ]:
import os
import sys

assert sys.version_info >= (3, 11), 'Use Python 3.11 ou superior'
sys.dont_write_bytecode = True
for key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[key] = '1'


## 1. Implementação
Execute a próxima célula sem alterações para disponibilizar todas as funções. Ela **não inicia experimentos**. Os métodos são implementações próprias de referência, não reproduções integrais dos artigos. O hash de código dos resultados identifica a implementação efetivamente executada nesta célula.

In [ ]:
# Implementação completa deste notebook: dados, heurísticas, solver, exportação e gráficos.
import csv
import hashlib
import inspect
import io
import json
import linecache
import math
import os
import platform
import random
import re
import statistics
import subprocess
import sys
import tarfile
import tempfile
import time
import traceback
import urllib.request
import zipfile
from collections import defaultdict
from dataclasses import asdict, dataclass, field
from datetime import UTC, datetime
from pathlib import Path, PurePosixPath

_CORE_SOURCE = "".join(linecache.getlines(inspect.currentframe().f_code.co_filename))
DATA_DIR = Path.cwd()
METHODS = ('greedy', 'greedy_vnd', 'grasp', 'ils', 'tabu', 'sa')
NEIGHBORHOODS = ('relocate', 'swap', 'open', 'close', 'replace')
PARQUET_FORMAT = 'sscflp.instances.v1'


def canonical_hash(value):
    raw = json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


@dataclass(frozen=True)
class Instance:
    name: str
    family: str
    origin: str
    capacities: tuple[float, ...]
    opening_costs: tuple[float, ...]
    demands: tuple[float, ...]
    assignment_costs: tuple[tuple[float, ...], ...]
    provenance: dict = field(default_factory=dict)

    def __post_init__(self):
        if not isinstance(self.name, str) or not self.name.strip():
            raise ValueError("A instância precisa de nome não vazio")
        if not isinstance(self.family, str) or not self.family.strip():
            raise ValueError("A instância precisa de família")
        if self.origin not in {"real", "synthetic", "derived", "example"}:
            raise ValueError("Origem inválida")
        if not isinstance(self.provenance, dict):
            raise TypeError("Proveniência deve ser um objeto JSON")
        m, n = len(self.capacities), len(self.demands)
        if not m or not n or len(self.opening_costs) != m:
            raise ValueError("Dimensões inválidas")
        if len(self.assignment_costs) != m or any(len(row) != n for row in self.assignment_costs):
            raise ValueError("Matriz deve ser instalação × cliente")
        vectors = (self.capacities, self.opening_costs, self.demands, *self.assignment_costs)
        if any(isinstance(v, bool) or not isinstance(v, (int, float))
               or not math.isfinite(v) or v < 0 for row in vectors for v in row):
            raise ValueError("Valores devem ser números finitos não negativos")
        if self.provenance.get("cost_basis") != "total_per_customer":
            raise ValueError("Normalizar e declarar cost_basis=total_per_customer")

    @classmethod
    def from_dict(cls, data):
        return cls(
            name=data["name"], family=data["family"], origin=data["origin"],
            capacities=tuple(data["capacities"]), opening_costs=tuple(data["opening_costs"]),
            demands=tuple(data["demands"]),
            assignment_costs=tuple(tuple(row) for row in data["assignment_costs"]),
            provenance=data.get("provenance", {}),
        )

    def to_dict(self):
        return asdict(self)

    @property
    def digest(self):
        return canonical_hash(self.to_dict())


@dataclass(frozen=True)
class Solution:
    assignment: tuple[int, ...]
    open_facilities: tuple[int, ...]
    loads: tuple[float, ...]
    opening_cost: float
    service_cost: float
    objective: float


def evaluate(instance, assignment):
    """Recompõe custo/cargas; não confia em métricas fornecidas pela busca."""
    assignment = tuple(assignment)
    m, n = len(instance.capacities), len(instance.demands)
    if len(assignment) != n or any(type(i) is not int or not 0 <= i < m for i in assignment):
        raise ValueError("Atribuição deve indicar um local válido por cliente")
    loads = [0.0] * m
    for j, i in enumerate(assignment):
        loads[i] += instance.demands[j]
    if any(load > cap + 1e-8 * max(1.0, cap) for load, cap in zip(loads, instance.capacities)):
        raise ValueError("Capacidade excedida")
    opened = tuple(sorted(set(assignment)))
    opening = math.fsum(instance.opening_costs[i] for i in opened)
    service = math.fsum(instance.assignment_costs[i][j] for j, i in enumerate(assignment))
    return Solution(assignment, opened, tuple(loads), opening, service, opening + service)


def parquet_modules():
    try:
        import pyarrow as pa
        import pyarrow.parquet as pq
    except ImportError as exc:
        raise RuntimeError("Instale o leitor: python -m pip install pyarrow") from exc
    return pa, pq


def parquet_schema(pa):
    vector = pa.list_(pa.field("element", pa.float64(), nullable=False))
    matrix = pa.list_(pa.field("element", vector, nullable=False))
    return pa.schema([
        pa.field("name", pa.string(), nullable=False),
        pa.field("family", pa.string(), nullable=False),
        pa.field("origin", pa.string(), nullable=False),
        pa.field("capacities", vector, nullable=False),
        pa.field("opening_costs", vector, nullable=False),
        pa.field("demands", vector, nullable=False),
        pa.field("assignment_costs", matrix, nullable=False),
        pa.field("provenance_json", pa.string(), nullable=False),
        pa.field("instance_sha256", pa.string(), nullable=False),
    ], metadata={b"sscflp_format": PARQUET_FORMAT.encode()})


def instances_hash(instances):
    if len({instance.name for instance in instances}) != len(instances):
        raise ValueError("Nomes de instâncias duplicados")
    return canonical_hash({instance.name: instance.digest for instance in instances})


def write_parquet(path, instances):
    """Uma linha por instância; vetores/matriz numéricos em float64, sem JSON de dados."""
    pa, pq = parquet_modules()
    instances = sorted(instances, key=lambda instance: instance.name)
    if not instances:
        raise ValueError("Fonte sem instâncias utilizáveis")
    digest = instances_hash(instances)
    rows = []
    for instance in instances:
        row = instance.to_dict()
        row["provenance_json"] = json.dumps(row.pop("provenance"), sort_keys=True, ensure_ascii=False)
        row["instance_sha256"] = instance.digest
        rows.append(row)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("xb") as handle:
        pq.write_table(pa.Table.from_pylist(rows, schema=parquet_schema(pa)), handle,
                       compression="zstd", version="2.6", row_group_size=1,
                       use_dictionary=["name", "family", "origin"])
    return {"file": path.name, "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
            "bytes": path.stat().st_size, "rows": len(instances), "instances_sha256": digest,
            "schema": PARQUET_FORMAT, "compression": "zstd", "pyarrow_version": pa.__version__}


def read_sources(directory=None):
    directory = DATA_DIR if directory is None else Path(directory)
    catalog = json.loads((directory / "origens.json").read_text(encoding="utf-8"))
    if catalog.get("schema_version") != 1 or catalog.get("data_format") != PARQUET_FORMAT:
        raise ValueError("Catálogo de fontes incompatível")
    sources = catalog.get("sources", [])
    filenames = [source["parquet"]["file"] for source in sources]
    if not sources or len(filenames) != len(set(filenames)) or len({s["family"] for s in sources}) != len(sources):
        raise ValueError("Fontes vazias ou duplicadas")
    for source in sources:
        name = source["parquet"]["file"]
        if Path(name).name != name or "\\" in name or not name.endswith(".parquet"):
            raise ValueError("Nome Parquet inválido no catálogo")
        if source["parquet"]["rows"] != source["original_instances"] - len(source["excluded"]):
            raise ValueError("Contagem de instâncias inconsistente no catálogo")
        if source["parquet"]["schema"] != PARQUET_FORMAT:
            raise ValueError("Versão do formato Parquet incompatível")
    if (catalog["original_instances"] != sum(source["original_instances"] for source in sources)
            or catalog["available_instances"] != sum(source["parquet"]["rows"] for source in sources)):
        raise ValueError("Totais do catálogo inconsistentes")
    return catalog


def load_instances(directory=None):
    """Lê somente os Parquet catalogados, verificando hashes, esquema e instâncias."""
    directory = DATA_DIR if directory is None else Path(directory)
    catalog = read_sources(directory)
    expected = {source["parquet"]["file"] for source in catalog["sources"]}
    if {path.name for path in directory.glob("*.parquet")} != expected:
        raise ValueError("Arquivos Parquet ausentes ou não catalogados; confira os quatro arquivos do Dataset")
    pa, pq = parquet_modules()
    instances = []
    for source in catalog["sources"]:
        description = source["parquet"]
        raw = (directory / description["file"]).read_bytes()
        if hashlib.sha256(raw).hexdigest() != description["sha256"]:
            raise ValueError(f"Parquet divergente: {description['file']}")
        table = pq.read_table(pa.BufferReader(raw))
        if (not table.schema.equals(parquet_schema(pa), check_metadata=False)
                or (table.schema.metadata or {}).get(b"sscflp_format") != PARQUET_FORMAT.encode()):
            raise ValueError(f"Esquema Parquet incompatível: {description['file']}")
        if table.num_rows != description["rows"]:
            raise ValueError("Contagem de linhas divergente")
        current = []
        for row in table.to_pylist():
            row["provenance"] = json.loads(row.pop("provenance_json"))
            instance = Instance.from_dict(row)
            if (instance.digest != row["instance_sha256"] or instance.family != source["family"]
                    or instance.origin != source["origin"]
                    or instance.provenance["archive_sha256"] != source["original"]["sha256"]
                    or instance.provenance["source"] != source["original"]["url"]
                    or instance.provenance["reference"] != source["reference"]["bibtex_key"]):
                raise ValueError(f"Instância/proveniência divergente: {instance.name}")
            current.append(instance)
        if instances_hash(current) != description["instances_sha256"]:
            raise ValueError("Hash semântico da fonte divergente")
        if {instance.name for instance in current} & {entry["name"] for entry in source["excluded"]}:
            raise ValueError("Instância em quarentena incluída nos dados")
        instances.extend(current)
    instances_hash(instances)
    return sorted(instances, key=lambda instance: instance.name)


def code_hash():
    if not _CORE_SOURCE:
        raise RuntimeError("Execute a implementação em um kernel Jupyter/IPython")
    return hashlib.sha256(_CORE_SOURCE.encode("utf-8")).hexdigest()


@dataclass(frozen=True)
class SearchConfig:
    time_limit: float = 1.0
    max_iterations: int | None = None
    initial_attempts: int = 32
    alpha: float = 0.3
    neighborhoods: tuple[str, ...] = NEIGHBORHOODS
    perturbation_moves: int = 3
    tabu_tenure: int = 7
    tabu_sample: int = 100
    temperature_fraction: float = 0.05
    cooling: float = 0.995

    def __post_init__(self):
        if any(not math.isfinite(v) for v in
               (self.time_limit, self.alpha, self.temperature_fraction, self.cooling)):
            raise ValueError("Parâmetros devem ser finitos")
        if self.time_limit <= 0 or not 0 <= self.alpha <= 1:
            raise ValueError("Tempo positivo e alpha em [0,1] são obrigatórios")
        for value in (self.initial_attempts, self.perturbation_moves, self.tabu_tenure, self.tabu_sample):
            if type(value) is not int or value < 1:
                raise ValueError("Contagens precisam ser inteiros positivos")
        if self.max_iterations is not None and (type(self.max_iterations) is not int or self.max_iterations < 1):
            raise ValueError("max_iterations deve ser inteiro positivo ou None")
        if not 0 < self.cooling < 1 or self.temperature_fraction <= 0:
            raise ValueError("Resfriamento em (0,1) e temperatura positiva")
        if not self.neighborhoods or any(k not in NEIGHBORHOODS for k in self.neighborhoods):
            raise ValueError("Vizinhanças inválidas")


class Tracker:
    def __init__(self, start, deadline):
        self.start, self.deadline = start, deadline
        self.best, self.history = None, []

    def offer(self, solution):
        if solution is not None and (self.best is None or solution.objective < self.best.objective - 1e-9):
            self.best = solution
            self.history.append({"seconds": time.perf_counter() - self.start, "objective": solution.objective})

    def active(self):
        return time.perf_counter() < self.deadline


def construct(instance, rng, deadline, alpha=0.0, allowed=None, order_mode=0):
    """Construção incremental; falha não prova inviabilidade."""
    if time.perf_counter() >= deadline:
        return None
    m, n = len(instance.capacities), len(instance.demands)
    allowed = tuple(range(m)) if allowed is None else tuple(sorted(allowed))
    if not allowed or sum(instance.capacities[i] for i in allowed) + 1e-8 < sum(instance.demands):
        return None
    remaining, opened, assignment = list(instance.capacities), set(), [-1] * n
    if order_mode == 0:
        order = sorted(range(n), key=lambda j: (-instance.demands[j], j))
    else:
        priorities = [instance.demands[j] * (0.5 + rng.random()) for j in range(n)]
        order = sorted(range(n), key=lambda j: (-priorities[j], j))
    for j in order:
        if time.perf_counter() >= deadline:
            return None
        choices = []
        for i in allowed:
            if remaining[i] + 1e-8 * max(1.0, instance.capacities[i]) >= instance.demands[j]:
                increment = instance.assignment_costs[i][j]
                if i not in opened:
                    increment += instance.opening_costs[i]
                choices.append((increment, i))
        if not choices:
            return None
        low, high = min(c for c, _ in choices), max(c for c, _ in choices)
        if alpha == 0:
            _, i = min(choices)
        else:
            rcl = [i for c, i in choices if c <= low + alpha * (high - low) + 1e-9]
            i = rng.choice(rcl)
        assignment[j] = i
        remaining[i] -= instance.demands[j]
        opened.add(i)
    return _evaluated(instance, assignment)


def _evaluated(instance, assignment):
    try:
        return evaluate(instance, assignment)
    except ValueError:
        return None


def neighbors(instance, solution, kind, rng, deadline, alpha):
    """Relocação/troca enumeradas; movimentos de instalações com reconstrução."""
    m, n = len(instance.capacities), len(instance.demands)
    assignment = solution.assignment
    if kind == "relocate":
        for j in range(n):
            for i in range(m):
                if time.perf_counter() >= deadline:
                    return
                if i != assignment[j]:
                    candidate = list(assignment)
                    candidate[j] = i
                    value = _evaluated(instance, candidate)
                    if value is not None:
                        yield value
    elif kind == "swap":
        for j in range(n):
            for k in range(j + 1, n):
                if time.perf_counter() >= deadline:
                    return
                if assignment[j] != assignment[k]:
                    candidate = list(assignment)
                    candidate[j], candidate[k] = candidate[k], candidate[j]
                    value = _evaluated(instance, candidate)
                    if value is not None:
                        yield value
    else:
        opened = set(solution.open_facilities)
        closed = sorted(set(range(m)) - opened)
        if kind == "open":
            subsets = (opened | {i} for i in closed)
        elif kind == "close":
            subsets = (opened - {i} for i in sorted(opened))
        elif kind == "replace":
            subsets = ((opened - {i}) | {k} for i in sorted(opened) for k in closed)
        else:
            raise ValueError(f"Vizinhança desconhecida: {kind}")
        for subset in subsets:
            if time.perf_counter() >= deadline:
                return
            value = construct(instance, rng, deadline, alpha=alpha, allowed=subset)
            if value is not None and value.assignment != assignment:
                yield value


def vnd(instance, solution, rng, config, tracker):
    index = 0
    while index < len(config.neighborhoods) and tracker.active():
        improved = False
        for candidate in neighbors(instance, solution, config.neighborhoods[index], rng,
                                   tracker.deadline, config.alpha):
            if candidate.objective < solution.objective - 1e-9:
                solution = candidate
                tracker.offer(solution)
                improved = True
                break
        index = 0 if improved else index + 1
    return solution


def random_neighbor(instance, solution, rng, deadline):
    n, m = len(instance.demands), len(instance.capacities)
    for _ in range(12):
        if time.perf_counter() >= deadline:
            return None
        candidate = list(solution.assignment)
        j = rng.randrange(n)
        if rng.random() < 0.5 and n > 1:
            k = rng.randrange(n)
            candidate[j], candidate[k] = candidate[k], candidate[j]
        else:
            candidate[j] = rng.randrange(m)
        if tuple(candidate) == solution.assignment:
            continue
        value = _evaluated(instance, candidate)
        if value is not None:
            return value
    return None


def run_search(instance, method, seed, config):
    if method not in METHODS:
        raise ValueError(f"Método inválido: {method}")
    start = time.perf_counter()
    tracker, rng = Tracker(start, start + config.time_limit), random.Random(seed)
    iterations, current = 0, None
    for attempt in range(config.initial_attempts):
        current = construct(instance, rng, tracker.deadline,
                            alpha=0.0 if attempt == 0 else config.alpha,
                            order_mode=0 if attempt == 0 else 1)
        if current is not None:
            tracker.offer(current)
            break
        if not tracker.active():
            break
    if current is None:
        return {"status": "no_feasible_found", "solution": None, "history": [],
                "elapsed_seconds": time.perf_counter() - start,
                "time_to_best_seconds": None, "iterations": 0}
    if method in {"greedy_vnd", "grasp", "ils"}:
        current = vnd(instance, current, rng, config, tracker)
    if method in {"greedy", "greedy_vnd"}:
        status = "completed" if tracker.active() else "time_limit"
    else:
        tabu = {}
        temperature = max(current.objective, 1.0) * config.temperature_fraction
        floor = temperature * 1e-6
        while tracker.active() and (config.max_iterations is None or iterations < config.max_iterations):
            iterations += 1
            if method == "grasp":
                candidate = construct(instance, rng, tracker.deadline, config.alpha, order_mode=1)
                if candidate is not None:
                    tracker.offer(candidate)
                    current = vnd(instance, candidate, rng, config, tracker)
            elif method == "ils":
                candidate = current
                if iterations % 10 == 0:
                    candidate = construct(instance, rng, tracker.deadline, config.alpha, order_mode=1) or current
                else:
                    for _ in range(config.perturbation_moves):
                        candidate = random_neighbor(instance, candidate, rng, tracker.deadline) or candidate
                        tracker.offer(candidate)
                tracker.offer(candidate)
                candidate = vnd(instance, candidate, rng, config, tracker)
                if candidate.objective < current.objective - 1e-9:
                    current = candidate
            elif method == "tabu":
                tabu = {key: expiry for key, expiry in tabu.items() if expiry > iterations}
                chosen = None
                for _ in range(config.tabu_sample):
                    candidate = random_neighbor(instance, current, rng, tracker.deadline)
                    if candidate is None:
                        if not tracker.active():
                            break
                        continue
                    changed = [j for j, i in enumerate(candidate.assignment) if i != current.assignment[j]]
                    forbidden = any(tabu.get((j, candidate.assignment[j]), 0) > iterations for j in changed)
                    aspiration = candidate.objective < tracker.best.objective - 1e-9
                    if (not forbidden or aspiration) and (chosen is None or candidate.objective < chosen.objective):
                        chosen = candidate
                if chosen is not None:
                    for j, i in enumerate(chosen.assignment):
                        if i != current.assignment[j]:
                            tabu[j, current.assignment[j]] = iterations + config.tabu_tenure + 1
                    current = chosen
                else:
                    current = construct(instance, rng, tracker.deadline, config.alpha, order_mode=1) or current
                tracker.offer(current)
            else:
                candidate = random_neighbor(instance, current, rng, tracker.deadline)
                if candidate is not None:
                    delta = candidate.objective - current.objective
                    if delta <= 0 or rng.random() < math.exp(-delta / temperature):
                        current = candidate
                        tracker.offer(current)
                temperature = max(floor, temperature * config.cooling)
        status = "time_limit" if not tracker.active() else "iteration_limit"
    best = evaluate(instance, tracker.best.assignment)
    return {"status": status, "solution": asdict(best), "history": tracker.history,
            "elapsed_seconds": time.perf_counter() - start,
            "time_to_best_seconds": tracker.history[-1]["seconds"], "iterations": iterations}


def utc_now():
    return datetime.now(UTC).isoformat()


def environment():
    cpu, memory = platform.processor(), None
    cpuinfo, meminfo = Path("/proc/cpuinfo"), Path("/proc/meminfo")
    if cpuinfo.exists():
        cpu = next((line.split(":", 1)[1].strip() for line in cpuinfo.read_text().splitlines()
                    if line.startswith("model name")), cpu)
    if meminfo.exists():
        memory = next((line.split(":", 1)[1].strip() for line in meminfo.read_text().splitlines()
                       if line.startswith("MemTotal")), None)
    result = {"python": platform.python_version(), "platform": platform.platform(),
              "pyarrow": getattr(sys.modules.get("pyarrow"), "__version__", None),
              "cpu": cpu, "logical_cpus": os.cpu_count(), "memory": memory,
              "threads": {key: os.environ.get(key) for key in
                          ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS")}}
    result["id"] = canonical_hash(result)
    return result


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
                         encoding="utf-8")
    temporary.replace(path)


def instance_metadata(instance):
    return {"name": instance.name, "family": instance.family, "origin": instance.origin,
            "facilities": len(instance.capacities), "customers": len(instance.demands),
            "provenance": json.loads(json.dumps(instance.provenance))}


def audit_record(record, instance):
    if record.get("schema_version") != 1 or record["instance_sha256"] != instance.digest:
        raise ValueError("Esquema/dados incompatíveis")
    identity = {key: record[key] for key in
                ("instance_sha256", "method", "seed", "config", "environment_id", "code_sha256", "stage")}
    if record.get("run_id") != canonical_hash(identity):
        raise ValueError("Identificador de execução inconsistente")
    if record["method"] not in METHODS or record["stage"] not in {"smoke", "pilot", "final"}:
        raise ValueError("Método/etapa inválido")
    SearchConfig(**{**record["config"], "neighborhoods": tuple(record["config"]["neighborhoods"])})
    env_payload = {k: v for k, v in record["environment"].items() if k != "id"}
    if record["environment_id"] != record["environment"]["id"] or canonical_hash(env_payload) != record["environment_id"]:
        raise ValueError("Ambiente inconsistente")
    if record["instance"] != instance_metadata(instance):
        raise ValueError("Metadados da instância divergem da original")
    result = record["result"]
    status = result["status"]
    if status not in {"completed", "time_limit", "iteration_limit", "no_feasible_found", "error"}:
        raise ValueError("Status desconhecido")
    if status == "error":
        if result.get("solution") is not None or not result.get("error"):
            raise ValueError("Registro de erro inconsistente")
        return
    elapsed = result["elapsed_seconds"]
    if not isinstance(elapsed, (int, float)) or not math.isfinite(elapsed) or elapsed < 0:
        raise ValueError("Tempo inválido")
    raw_solution = result["solution"]
    if raw_solution is None:
        if status != "no_feasible_found" or result["history"] or result["time_to_best_seconds"] is not None:
            raise ValueError("Solução ausente sem status correspondente")
        return
    if status == "no_feasible_found":
        raise ValueError("Status sem viável contém solução")
    checked = evaluate(instance, raw_solution["assignment"])
    for key in ("objective", "opening_cost", "service_cost"):
        value = raw_solution[key]
        if not math.isfinite(value) or not math.isclose(value, getattr(checked, key), rel_tol=1e-9, abs_tol=1e-7):
            raise ValueError(f"Custo inconsistente: {key}")
    if tuple(raw_solution["open_facilities"]) != checked.open_facilities:
        raise ValueError("Locais abertos inconsistentes")
    if len(raw_solution["loads"]) != len(checked.loads) or any(
        not math.isclose(a, b, rel_tol=1e-9, abs_tol=1e-7)
        for a, b in zip(raw_solution["loads"], checked.loads)
    ):
        raise ValueError("Cargas inconsistentes")
    history = result["history"]
    previous_time, previous_cost = -1.0, math.inf
    for point in history:
        t, cost = point["seconds"], point["objective"]
        if not math.isfinite(t) or not math.isfinite(cost) or not previous_time <= t <= elapsed or cost >= previous_cost:
            raise ValueError("Histórico inválido")
        previous_time, previous_cost = t, cost
    if not history or not math.isclose(previous_cost, checked.objective, rel_tol=1e-9, abs_tol=1e-7):
        raise ValueError("Histórico não termina na melhor solução")
    if result["time_to_best_seconds"] != history[-1]["seconds"]:
        raise ValueError("Tempo até a melhor solução inconsistente")


def run_batch(instances, methods, seeds, config, output_dir, *, stage="smoke", label="unnamed",
              retry_failures=False, progress=print, after_each=None):
    instances, methods, seeds = list(instances), list(methods), list(seeds)
    if stage not in {"smoke", "pilot", "final"}:
        raise ValueError("Etapa inválida")
    if not instances or not methods or not seeds or any(m not in METHODS for m in methods):
        raise ValueError("Lote vazio ou método inválido")
    if len(set(methods)) != len(methods) or len(set(seeds)) != len(seeds):
        raise ValueError("Métodos/sementes duplicados")
    if len({i.name for i in instances}) != len(instances) or any(type(seed) is not int for seed in seeds):
        raise ValueError("Instâncias duplicadas ou sementes não inteiras")
    if stage == "final" and any(i.origin == "example" or
                               i.provenance.get("interpretation_status") != "reviewed" for i in instances):
        raise ValueError("Avaliação final exige dados revisados e não didáticos")
    output = Path(output_dir)
    source_hash, env = code_hash(), environment()
    batch = {"label": label, "created_utc": utc_now(), "environment": env,
             "code_sha256": source_hash, "stage": stage, "methods": methods,
             "seeds": seeds, "config": asdict(config),
             "instances": [{"name": i.name, "sha256": i.digest} for i in instances]}
    atomic_json(output / "batches" / f"{canonical_hash(batch)}.json", batch)
    paths = []
    for instance in instances:
        for method in methods:
            for seed in seeds:
                identity = {"instance_sha256": instance.digest, "method": method, "seed": seed,
                            "config": asdict(config), "environment_id": env["id"],
                            "code_sha256": source_hash, "stage": stage}
                run_id = canonical_hash(identity)
                path = output / "runs" / f"{run_id}.json"
                if path.exists():
                    old = json.loads(path.read_text(encoding="utf-8"))
                    audit_record(old, instance)
                    if not retry_failures or old["result"]["status"] not in {"error", "no_feasible_found"}:
                        progress(f"Retomada: {instance.name} / {method} / {seed}")
                        paths.append(path)
                        continue
                    atomic_json(output / "attempts" / f"{run_id}-{canonical_hash(old)}.json", old)
                record = {"schema_version": 1, "run_id": run_id, **identity,
                          "created_utc": utc_now(), "environment": env, "instance": instance_metadata(instance)}
                try:
                    record["result"] = run_search(instance, method, seed, config)
                    audit_record(record, instance)
                except Exception:  # noqa: BLE001 -- persistir falha sem perder o lote
                    record["result"] = {"status": "error", "solution": None, "error": traceback.format_exc()}
                atomic_json(path, record)
                paths.append(path)
                progress(f"{instance.name} / {method} / {seed}: {record['result']['status']}")
                if after_each is not None:
                    after_each(output)
    return paths


def summarize(results_dir, data_source, destination):
    known = {i.digest: i for i in load_instances(data_source)}
    records = {}
    for path, record in result_records(results_dir, "runs"):
        instance = known.get(record.get("instance_sha256"))
        if instance is None:
            raise ValueError(f"Instância desconhecida em {path}")
        audit_record(record, instance)
        run_id = record["run_id"]
        if run_id in records and records[run_id] != record:
            raise ValueError(f"Execução duplicada divergente: {run_id}")
        records[run_id] = record
    if not records:
        raise ValueError("Nenhum resultado em runs/*.json")
    prior_attempts = {}
    for path, record in result_records(results_dir, "attempts"):
        instance = known.get(record.get("instance_sha256"))
        if instance is None:
            raise ValueError(f"Instância desconhecida: {path}")
        audit_record(record, instance)
        if record["run_id"] not in records:
            raise ValueError("Tentativa anterior sem registro corrente")
        digest = canonical_hash(record)
        if digest != canonical_hash(records[record["run_id"]]):
            prior_attempts[record["run_id"], digest] = record
    groups = defaultdict(list)
    for record in records.values():
        key = (record["instance_sha256"], record["method"], record["environment_id"],
               record["code_sha256"], record["stage"],
               json.dumps(record["config"], sort_keys=True, separators=(",", ":")))
        groups[key].append(record)
    rows = []
    for key, members in sorted(groups.items()):
        results = [m["result"] for m in members]
        run_ids = {m["run_id"] for m in members}
        previous = [r["result"] for r in prior_attempts.values() if r["run_id"] in run_ids]
        objectives = [r["solution"]["objective"] for r in results if r.get("solution") is not None]
        times = [r["elapsed_seconds"] for r in results if "elapsed_seconds" in r]
        instance = known[key[0]]
        rows.append({"instance": instance.name, "family": instance.family, "origin": instance.origin,
                     "instance_sha256": key[0], "method": key[1], "environment_id": key[2],
                     "code_sha256": key[3], "stage": key[4], "config_json": key[5],
                     "attempted": len(results), "feasible": len(objectives),
                     "prior_attempts": len(previous), "total_attempts": len(results) + len(previous),
                     "prior_errors": sum(r["status"] == "error" for r in previous),
                     "prior_no_feasible_found": sum(r["status"] == "no_feasible_found" for r in previous),
                     "errors": sum(r["status"] == "error" for r in results),
                     "no_feasible_found": sum(r["status"] == "no_feasible_found" for r in results),
                     "best": min(objectives) if objectives else None,
                     "mean": statistics.mean(objectives) if objectives else None,
                     "median": statistics.median(objectives) if objectives else None,
                     "stdev": statistics.stdev(objectives) if len(objectives) > 1 else None,
                     "mean_elapsed_seconds": statistics.mean(times) if times else None})
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    return rows


def result_records(source, category):
    source = Path(source)
    if source.is_dir():
        for path in sorted(source.rglob(f"{category}/*.json")):
            yield str(path), json.loads(path.read_text(encoding="utf-8"))
    else:
        with zipfile.ZipFile(source) as archive:
            if len(archive.namelist()) != len(set(archive.namelist())):
                raise ValueError("Resultados com entradas duplicadas")
            for name in sorted(archive.namelist()):
                parts = PurePosixPath(name).parts
                if len(parts) == 2 and parts[0] == category and name.endswith(".json"):
                    yield name, json.loads(archive.read(name))


def save_results(directory, destination):
    directory, destination = Path(directory), Path(destination)
    temporary = destination.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for category in ("runs", "batches", "attempts"):
            for path in sorted((directory / category).glob("*.json")):
                archive.write(path, path.relative_to(directory))
    temporary.replace(destination)


def restore_results(source, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(source) as archive:
        if len(archive.namelist()) != len(set(archive.namelist())):
            raise ValueError("Resultados com entradas duplicadas")
        for name in archive.namelist():
            parts = PurePosixPath(name).parts
            if len(parts) != 2 or parts[0] not in {"runs", "batches", "attempts"} or not name.endswith(".json"):
                raise ValueError(f"Entrada inesperada: {name}")
            path = (destination / name).resolve()
            if not path.is_relative_to(destination):
                raise ValueError("Caminho inseguro")
            data = archive.read(name)
            json.loads(data)
            if path.exists() and path.read_bytes() != data:
                raise ValueError(f"Conflito de resultados: {name}")
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(data)


def solve_instance(instance, destination, *, seconds=5.0, lp_seconds=None, seed=0):
    """Resolve LP/MIP localmente e entrega apenas um ZIP de logs e resultados."""
    try:
        import pyscipopt
        from pyscipopt import Model, quicksum
    except ImportError as exc:
        raise RuntimeError("Instale: python -m pip install pyscipopt==6.3.0") from exc
    lp_seconds = seconds if lp_seconds is None else lp_seconds
    if any(not math.isfinite(t) or t <= 0 for t in (seconds, lp_seconds)):
        raise ValueError("Limites LP/MIP devem ser positivos")
    if type(seed) is not int or not 0 <= seed <= 2147483647:
        raise ValueError("Semente SCIP inválida")
    destination = Path(destination)
    if destination.exists():
        raise FileExistsError(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    results, overall_start = {}, time.perf_counter()
    with tempfile.TemporaryDirectory(dir=destination.resolve().parent) as directory:
        output = Path(directory)
        for kind, limit in (("lp", lp_seconds), ("mip", seconds)):
            start = time.perf_counter()
            model = Model(f"{instance.name}-{kind}")
            scip_version = f"{model.getMajorVersion()}.{model.getMinorVersion()}.{model.getTechVersion()}"
            model.setLogfile(str(output / f"{kind}.log"))
            for key, value in {"limits/time": float(limit), "limits/gap": 0.0,
                               "limits/absgap": 0.0, "parallel/maxnthreads": 1,
                               "randomization/randomseedshift": seed}.items():
                model.setParam(key, value)
            vtype = "C" if kind == "lp" else "B"
            m, n = len(instance.capacities), len(instance.demands)
            y = [model.addVar(name=f"y_{i}", vtype=vtype, lb=0, ub=1) for i in range(m)]
            x = [[model.addVar(name=f"x_{i}_{j}", vtype=vtype, lb=0, ub=1)
                  for j in range(n)] for i in range(m)]
            model.setObjective(quicksum(instance.opening_costs[i] * y[i] for i in range(m)) +
                               quicksum(instance.assignment_costs[i][j] * x[i][j]
                                        for i in range(m) for j in range(n)), "minimize")
            for j in range(n):
                model.addCons(quicksum(x[i][j] for i in range(m)) == 1, name=f"assign_{j}")
            for i in range(m):
                model.addCons(quicksum(instance.demands[j] * x[i][j] for j in range(n)) <=
                              instance.capacities[i] * y[i], name=f"capacity_{i}")
                for j in range(n):
                    model.addCons(x[i][j] <= y[i], name=f"link_{i}_{j}")
            build_seconds = time.perf_counter() - start
            variables, constraints = model.getNVars(), model.getNConss()
            atomic_json(output / f"{kind}-parameters.json", model.getParams())
            model.optimize()

            def finite_scip(value, model=model):
                return float(value) if math.isfinite(value) and not model.isInfinity(abs(value)) else None

            status = str(model.getStatus())
            best_sol = model.getBestSol() if model.getNSols() else None
            objective = finite_scip(model.getSolObjVal(best_sol)) if best_sol is not None else None
            result = {"status": status, "limit_seconds": limit,
                      "original_variables": variables, "original_constraints": constraints,
                      "build_seconds": build_seconds, "solver_seconds": model.getSolvingTime(),
                      "wall_seconds": time.perf_counter() - start, "raw_objective": objective,
                      "dual_bound": finite_scip(model.getDualbound()),
                      "gap_fraction_scip": finite_scip(model.getGap()) if best_sol is not None else None,
                      "nodes": model.getNNodes(), "lp_iterations": model.getNLPIterations(), "solution": None}
            if kind == "lp":
                result["optimal_lp_value"] = objective if status == "optimal" else None
            elif best_sol is not None:
                assignment = []
                for j in range(n):
                    values = [model.getSolVal(best_sol, x[i][j]) for i in range(m)]
                    i = max(range(m), key=lambda k: values[k])
                    if values[i] < 1 - 1e-5:
                        raise ValueError("Extração não inteira")
                    assignment.append(i)
                checked = evaluate(instance, assignment)
                if objective is None or not math.isclose(checked.objective, objective, rel_tol=1e-7, abs_tol=1e-6):
                    raise ValueError("Objetivo independente diverge do SCIP")
                result["solution"] = asdict(checked)
            results[kind] = result
            model.freeProb()
        record = {"schema_version": 1, "created_utc": utc_now(), "instance": instance.to_dict(),
                  "instance_sha256": instance.digest, "code_sha256": code_hash(), "environment": environment(),
                  "pyscipopt_version": pyscipopt.__version__, "scip_version": scip_version, "seed": seed,
                  "total_wall_seconds": time.perf_counter() - overall_start, "results": results}
        atomic_json(output / "solver.json", record)
        with zipfile.ZipFile(destination, "x", compression=zipfile.ZIP_DEFLATED) as archive:
            for path in sorted(output.iterdir()):
                archive.write(path, path.name)
    return record


def parse_pisa(data, name, source, member):
    payload = data.rstrip(b"\x00 \t\r\n")
    removed_nul = data.count(b"\x00") - payload.count(b"\x00")
    if b"\x00" in payload:
        raise ValueError("NUL interno ou dados após preenchimento")
    tokens = payload.decode("utf-8").split()
    if len(tokens) < 2:
        raise ValueError("Cabeçalho ausente")
    dimensions = [float(token) for token in tokens[:2]]
    m, n = map(int, dimensions)
    if m <= 0 or n <= 0 or dimensions != [m, n] or len(tokens) != 2 + 2*m + n + m*n:
        raise ValueError("Contagem/dimensões incompatíveis com o layout de Pisa")
    values, offset = [float(token) for token in tokens], 2 + 2*m
    return Instance(
        name=name, family=source["family"], origin=source["origin"],
        capacities=tuple(values[2 + 2*i] for i in range(m)),
        opening_costs=tuple(values[3 + 2*i] for i in range(m)), demands=tuple(values[offset:offset+n]),
        assignment_costs=tuple(tuple(values[offset+n+i*n:offset+n+(i+1)*n]) for i in range(m)),
        provenance={"source": source["original"]["url"], "source_member": member,
                    "reference": source["reference"]["bibtex_key"],
                    "downloaded_utc": source["original"]["downloaded_utc"],
                    "archive_sha256": source["original"]["sha256"],
                    "original_sha256": hashlib.sha256(data).hexdigest(), "cost_basis": "total_per_customer",
                    "interpretation_status": "format_checked_pending_domain_review",
                    "layout": "m,n; capacity/fixed pairs; demand vector; facility-major cost matrix",
                    "cost_conversion": "none; coefficients preserved from the Pisa distribution",
                    "review_pending": "Confirm cost interpretation and units before final evaluation",
                    "redistribution": source["rights"]["redistribution"],
                    **({"trailing_nul_bytes_removed": removed_nul} if removed_nul else {})},
    )


def convert_original(data, source):
    """Usado apenas para repor Parquet; os originais ficam somente em memória."""
    filename = source["original"]["file"]
    if filename.endswith(".tar.Z"):
        expanded = subprocess.run(["gzip", "-dc"], input=data, capture_output=True, check=True).stdout
        with tarfile.open(fileobj=io.BytesIO(expanded), mode="r:") as archive:
            records = [(PurePosixPath(member.name).name, archive.extractfile(member).read(), member.name)
                       for member in archive.getmembers() if member.isfile()]
    elif filename.endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(data)) as archive:
            records = [(PurePosixPath(member.filename).name, archive.read(member), member.filename)
                       for member in archive.infolist() if not member.is_dir()]
    elif filename.endswith(".tex"):
        records = [(Path(filename).stem, data, filename)]
    else:
        raise ValueError("Formato original não suportado")
    if len(records) != source["original_instances"] or len({name for name, _, _ in records}) != len(records):
        raise ValueError("Inventário original divergente")
    expected_exclusions = {entry["name"]: entry for entry in source["excluded"]}
    instances, excluded = [], set()
    for name, raw, member in records:
        try:
            instance = parse_pisa(raw, name, source, member)
        except (ValueError, OverflowError):
            entry = expected_exclusions.get(name)
            if entry is None or entry["original_sha256"] != hashlib.sha256(raw).hexdigest():
                raise
            excluded.add(name)
            continue
        if name in expected_exclusions:
            raise ValueError("Uma exclusão conhecida não foi reproduzida")
        instances.append(instance)
    if excluded != set(expected_exclusions) or instances_hash(instances) != source["parquet"]["instances_sha256"]:
        raise ValueError("Conversão não reproduziu as instâncias registradas")
    return instances


def download_data(directory=None):
    """Repõe Parquet ausentes a partir das URLs do JSON, sem gravar arquivos raw."""
    directory = DATA_DIR if directory is None else Path(directory)
    catalog = read_sources(directory)
    for source in catalog["sources"]:
        path = directory / source["parquet"]["file"]
        if path.exists():
            if hashlib.sha256(path.read_bytes()).hexdigest() != source["parquet"]["sha256"]:
                raise ValueError(f"Parquet divergente, não sobrescrito: {path.name}")
            continue
        with urllib.request.urlopen(source["original"]["url"], timeout=90) as response:
            raw = response.read(100 * 1024 * 1024 + 1)
        if len(raw) > 100 * 1024 * 1024 or hashlib.sha256(raw).hexdigest() != source["original"]["sha256"]:
            raise ValueError("Download não corresponde ao original registrado")
        instances = convert_original(raw, source)
        temporary_root = "/tmp/opencode" if Path("/tmp/opencode").is_dir() else None
        with tempfile.TemporaryDirectory(dir=temporary_root) as folder:
            temporary = Path(folder) / path.name
            description = write_parquet(temporary, instances)
            with path.open("xb") as handle:
                handle.write(temporary.read_bytes())
        source["parquet"] = description
        source["conversion"]["converted_utc"] = utc_now()
        atomic_json(directory / "origens.json", catalog)
        print("Parquet recuperado:", path)
    load_instances(directory)


def generate_graphs(results, output, solver_files=(), data_dir=None):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError as exc:
        raise RuntimeError("Instale: python -m pip install matplotlib") from exc
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    summarize(results, data_dir or DATA_DIR, output / "resumo.csv")
    known = {instance.digest: instance for instance in load_instances(data_dir)}
    records = {record["run_id"]: record for _, record in result_records(results, "runs")}
    references = {}
    for path in solver_files:
        with zipfile.ZipFile(path) as archive:
            record = json.loads(archive.read("solver.json"))
        instance = known.get(record["instance_sha256"])
        if instance is None or Instance.from_dict(record["instance"]).digest != instance.digest:
            raise ValueError(f"Solver usa outros dados: {path}")
        mip = record["results"]["mip"]
        if mip["solution"] is not None:
            checked = evaluate(instance, mip["solution"]["assignment"])
            if not math.isclose(checked.objective, mip["solution"]["objective"], rel_tol=1e-9, abs_tol=1e-7):
                raise ValueError(f"Solução do solver inconsistente: {path}")
            if mip["status"] == "optimal":
                previous = references.get(instance.digest, checked.objective)
                if not math.isclose(previous, checked.objective, rel_tol=1e-7, abs_tol=1e-6):
                    raise ValueError("Ótimos de referência conflitantes")
                references[instance.digest] = checked.objective
        elif mip["status"] == "optimal":
            raise ValueError("Solver declarou ótimo sem solução")

    by_instance = defaultdict(lambda: defaultdict(list))
    for record in records.values():
        config_hash = canonical_hash(record["config"])
        key = (record["method"], record["stage"], record["environment_id"], record["code_sha256"], config_hash)
        by_instance[record["instance_sha256"]][key].append(record)
    written = []
    for digest, groups in sorted(by_instance.items()):
        instance = known[digest]
        prefix = re.sub(r"[^A-Za-z0-9_-]", "_", instance.name) + "-" + digest[:8]
        ordered = sorted(groups.items())
        labels = [f"{key[0]} [{key[1]}]\n{values[0]['config']['time_limit']:g}s · cfg {key[4][:4]}\n"
                  f"amb {key[2][:4]} · cod {key[3][:4]}" for key, values in ordered]
        reference = references.get(digest)
        relative = reference is not None and reference != 0

        def quality(value, reference=reference, relative=relative):
            return 100 * (value - reference) / abs(reference) if relative else value

        ylabel = "Gap para ótimo comprovado (%)" if relative else "Custo da solução"
        width = max(8, min(24, len(labels) * 2))

        def save(fig, kind, prefix=prefix, digest=digest):
            fig.tight_layout()
            path = output / f"{prefix}-{kind}.png"
            fig.savefig(path, dpi=160, metadata={
                "Description": f"instancia={digest}; matplotlib={matplotlib.__version__}; "
                               f"analise={code_hash()}"})
            plt.close(fig)
            written.append(path)

        objectives, objective_labels, durations, duration_labels = [], [], [], []
        for label, (_, members) in zip(labels, ordered):
            costs = [quality(record["result"]["solution"]["objective"])
                     for record in members if record["result"].get("solution") is not None]
            times = [record["result"]["elapsed_seconds"] for record in members
                     if "elapsed_seconds" in record["result"]]
            if costs:
                objectives.append(costs)
                objective_labels.append(label)
            if times:
                durations.append(times)
                duration_labels.append(label)
        if objectives:
            fig, ax = plt.subplots(figsize=(width, 5))
            ax.boxplot(objectives, tick_labels=objective_labels, showmeans=True)
            ax.set(ylabel=ylabel, title=f"{instance.name} — qualidade entre execuções viáveis")
            ax.grid(axis="y", alpha=0.25)
            save(fig, "qualidade")
        if durations:
            fig, ax = plt.subplots(figsize=(width, 5))
            ax.boxplot(durations, tick_labels=duration_labels, showmeans=True)
            ax.set(ylabel="Tempo real de busca (s)", title=f"{instance.name} — tempos por ambiente")
            ax.grid(axis="y", alpha=0.25)
            save(fig, "tempo")

        fig, ax = plt.subplots(figsize=(width, 5))
        bottom = [0] * len(ordered)
        for status in ("completed", "time_limit", "iteration_limit", "no_feasible_found", "error"):
            counts = [sum(record["result"]["status"] == status for record in members) for _, members in ordered]
            if any(counts):
                ax.bar(range(len(ordered)), counts, bottom=bottom, label=status)
                bottom = [a + b for a, b in zip(bottom, counts)]
        ax.set_xticks(range(len(labels)), labels)
        ax.set(ylabel="Execuções correntes (uma por semente/configuração)", title=f"{instance.name} — status")
        ax.legend(fontsize="small")
        save(fig, "status")

        if objectives:
            fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
            for label, (_, members) in zip(labels, ordered):
                limit = members[0]["config"]["time_limit"]
                grid = [limit * index / 100 for index in range(101)]
                means, rates = [], []
                for instant in grid:
                    values = []
                    for record in members:
                        past = [point["objective"] for point in record["result"].get("history", [])
                                if point["seconds"] <= instant]
                        if past:
                            values.append(quality(past[-1]))
                    means.append(statistics.mean(values) if values else math.nan)
                    rates.append(100 * len(values) / len(members))
                axes[0].plot(grid, means, label=label.replace("\n", " / "))
                axes[1].plot(grid, rates)
            axes[0].set(ylabel=ylabel, title=f"{instance.name} — média entre execuções com solução no instante")
            axes[0].legend(fontsize="x-small")
            axes[1].set(xlabel="Tempo de busca (s)", ylabel="Execuções com solução (%)", ylim=(-2, 102))
            for ax in axes:
                ax.grid(alpha=0.25)
            save(fig, "convergencia")
    print(f"CSV auditado e {len(written)} gráficos em {output}")
    print("Ambientes/configurações separados. Tempos de máquinas distintas não são speedup controlado.")
    return written


In [ ]:
DATA_PATH = None  # automático; informe uma pasta somente se houver mais de um Dataset com origens.json
if DATA_PATH is None:
    input_root = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd()
    candidates = sorted({path.parent.resolve() for path in input_root.rglob('origens.json')})
    if len(candidates) != 1:
        raise ValueError('Anexe um único conjunto com origens.json ou preencha DATA_PATH com sua pasta')
    DATA_PATH = candidates[0]
DATA_DIR = Path(DATA_PATH)
instances = load_instances(DATA_DIR)
print('Pasta dos dados:', DATA_DIR)
print('Instâncias disponíveis:', len(instances), '| Código:', code_hash())
print(json.dumps(environment(), indent=2, ensure_ascii=False))


## 2. Configuração
Habilite apenas o necessário: `RUN_HEURISTICS`, `RUN_SOLVER` ou `RUN_PLOTS`. Para iniciar, habilite **somente as heurísticas** com uma instância e uma semente; o teste inicial prevê até aproximadamente 30 s de busca. Isso verifica a plataforma, não substitui a avaliação científica.

Heurísticas demoradas: Kaggle. Solver e gráficos podem ser executados localmente, abrindo **este mesmo main.ipynb**. Para analisar resultados anteriores sem rodar heurísticas, preencha `RESULTS_ZIP` e habilite só `RUN_PLOTS`. Para retomar um lote, use `RESUME_ZIP`.

Separe calibração e teste, reserve `saiwa` se for o caso real final e registre o orçamento. O status `final` exige interpretação dos dados revisada; converter para Parquet não realiza essa revisão.

In [ ]:
RUN_HEURISTICS = False
RUN_SOLVER = False
RUN_PLOTS = False

LABEL = 'teste-01'
STAGE = 'pilot'
NAMES = ['p1']
METHODS_TO_RUN = ['greedy', 'greedy_vnd', 'grasp', 'ils', 'tabu', 'sa']
SEEDS = [0]
SEARCH = SearchConfig(time_limit=5.0)
RESUME_ZIP = ''
RETRY_FAILURES = False

SOLVER_NAMES = ['p1']
SOLVER_SECONDS = 5.0
LP_SECONDS = 5.0
SOLVER_SEED = 0

RESULTS_ZIP = ''  # vazio: usar o ZIP produzido pelo lote atual
SOLVER_ZIPS = []  # ZIPs anteriores do solver, opcionais para mostrar gaps aos ótimos comprovados

if not re.fullmatch(r'[A-Za-z0-9_-]+', LABEL):
    raise ValueError('LABEL inválido')
available_names = {instance.name for instance in instances}
if RUN_HEURISTICS and (not NAMES or len(set(NAMES)) != len(NAMES) or not set(NAMES) <= available_names):
    raise ValueError('Escolha instâncias distintas e disponíveis para as heurísticas')
if RUN_SOLVER and (not SOLVER_NAMES or len(set(SOLVER_NAMES)) != len(SOLVER_NAMES) or not set(SOLVER_NAMES) <= available_names):
    raise ValueError('Escolha instâncias distintas e disponíveis para o solver')
if RUN_HEURISTICS and (not METHODS_TO_RUN or not set(METHODS_TO_RUN) <= set(METHODS)):
    raise ValueError('Método desconhecido ou seleção vazia')
selected = [instance for instance in instances if instance.name in NAMES]
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('resultados')
OUTPUT = WORK / LABEL
EXPORT = WORK / f'{LABEL}.zip'
if RUN_HEURISTICS or RESUME_ZIP:
    OUTPUT.mkdir(parents=True, exist_ok=True)
print('Habilitados — heurísticas:', RUN_HEURISTICS, '| solver:', RUN_SOLVER, '| gráficos:', RUN_PLOTS)
print('Orçamento aproximado das heurísticas (h):', len(selected) * len(METHODS_TO_RUN) * len(SEEDS) * SEARCH.time_limit / 3600)
print('Limites do solver (s por instância): LP', LP_SECONDS, '+ MIP', SOLVER_SECONDS)


## 3. Heurísticas e retomada
O tempo inclui construção; a parada é cooperativa e pode exceder ligeiramente o limite. Cada execução concluída salva seu resultado e atualiza o ZIP. Uma busca interrompida recomeça; execuções completas com a mesma identidade são puladas. Mudar ambiente, código ou parâmetros produz outra identidade.

In [ ]:
if RESUME_ZIP:
    restore_results(RESUME_ZIP, OUTPUT)
    known = {instance.digest: instance for instance in instances}
    for _, record in result_records(OUTPUT, 'runs'):
        audit_record(record, known[record['instance_sha256']])
if RUN_HEURISTICS:
    try:
        run_batch(selected, METHODS_TO_RUN, SEEDS, SEARCH, OUTPUT, stage=STAGE, label=LABEL,
                  retry_failures=RETRY_FAILURES,
                  after_each=lambda directory: save_results(directory, EXPORT))
    finally:
        if any(OUTPUT.glob('runs/*.json')):
            save_results(OUTPUT, EXPORT)
            print('Baixe o ZIP de resultados:', EXPORT)
else:
    print('Heurísticas desligadas.')


## 4. Solver LP/MIP — opcional
Ative `RUN_SOLVER` somente após definir instâncias e limites. O solver SCIP é carregado apenas quando habilitado. Cada ZIP guarda solução, limites, status, nós, iterações, configuração e logs. LP e MIP têm limites separados. `timelimit` com incumbente não prova otimalidade. Arquivos existentes não são sobrescritos: mude `LABEL` para outra execução.

In [ ]:
generated_solver_zips = []
if RUN_SOLVER:
    for instance in instances:
        if instance.name not in SOLVER_NAMES:
            continue
        destination = WORK / f'{LABEL}-solver-{instance.name}.zip'
        solved = solve_instance(instance, destination, seconds=SOLVER_SECONDS,
                                lp_seconds=LP_SECONDS, seed=SOLVER_SEED)
        generated_solver_zips.append(destination)
        print(instance.name, {kind: result['status'] for kind, result in solved['results'].items()})
        print('Baixe o ZIP do solver:', destination)
else:
    print('Solver desligado.')


## 5. Gráficos e CSV — opcionais
Ative `RUN_PLOTS` para auditar resultados e gerar qualidade, tempo, status e convergência por instância. Podem ser resultados deste lote ou um ZIP anterior indicado em `RESULTS_ZIP`. `SOLVER_ZIPS` permite usar ótimos comprovados de execuções anteriores; ZIPs do solver produzidos com o mesmo `LABEL` também são encontrados.

Sem ótimo comprovado, qualidade é custo, não gap ao ótimo. A curva usa somente soluções obtidas até cada instante e apresenta a taxa de viáveis. Ambientes e configurações ficam separados; tempos de máquinas diferentes não são speedup controlado.

In [ ]:
if RUN_PLOTS:
    results_source = Path(RESULTS_ZIP) if RESULTS_ZIP else (EXPORT if EXPORT.is_file() else Path(RESUME_ZIP) if RESUME_ZIP else None)
    if results_source is None or not results_source.is_file():
        raise ValueError('Informe RESULTS_ZIP ou execute primeiro um lote de heurísticas')
    solver_sources = sorted({Path(path).resolve() for path in SOLVER_ZIPS}
                            | {path.resolve() for path in generated_solver_zips}
                            | {path.resolve() for path in WORK.glob(f'{LABEL}-solver-*.zip')})
    plot_dir = WORK / f'{LABEL}-graficos'
    images = generate_graphs(results_source, plot_dir, solver_sources, DATA_DIR)
    plot_zip = WORK / f'{LABEL}-graficos.zip'
    temporary = plot_zip.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted({plot_dir / 'resumo.csv', *images}):
            archive.write(path, path.name)
    temporary.replace(plot_zip)
    print('Baixe os gráficos e CSV:', plot_zip)
    from IPython.display import Image, display
    for path in images[:4]:
        display(Image(filename=str(path)))
else:
    print('Gráficos desligados.')


## Preservar a execução
Com `LABEL='teste-01'`, as saídas habilitadas ficam em `teste-01.zip` (heurísticas), `teste-01-solver-<instancia>.zip` e `teste-01-graficos.zip`. No Kaggle estão em `/kaggle/working`; localmente em `resultados/`. **Baixe os ZIPs e salve uma versão do notebook com as saídas.** Não dependa apenas da sessão temporária.

Para análise local, abra este mesmo `main.ipynb`, deixe heurísticas e solver desligados, informe o ZIP em `RESULTS_ZIP` e habilite só os gráficos. Para repor dados ausentes localmente, a função `download_data(DATA_DIR)` usa as URLs/hashes de `origens.json`; precisa de pasta gravável e gzip apenas para converter o original histórico Holmberg. Nenhuma recuperação ocorre automaticamente.

Fontes e protocolo: `origens.json`, `README.md`, `references.bib` e o enunciado do professor. A conversão de formato não substitui revisão de custos/unidades, e os testes iniciais não são resultados científicos finais.